In [1]:
from PIL import Image
import fitz
import os
import pandas as pd
import glob
from tqdm import tqdm

from pathlib import Path
from docling.backend.docling_parse_backend import DoclingParseDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    EasyOcrOptions,
    OcrMacOptions,
    PdfPipelineOptions,
    RapidOcrOptions,
    TesseractCliOcrOptions,
    TesseractOcrOptions,
)
from docling.datamodel.settings import settings
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.models.tesseract_ocr_cli_model import TesseractCliOcrOptions
from docling.models.tesseract_ocr_model import TesseractOcrOptions


import time

c:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def extrai_texto_de_imagem(image):
    # # Converter a imagem para preto e branco
    bw_image = image.convert("L")
    # # Salvar a imagem temporariamente para processamento
    temp_image_path = Path("C:/Estudos e Projetos/2024-LLM-Agronomo-Virtual/1 - criação_base/pasta_docs_tmp/temp_image.jpeg")
    bw_image.save(temp_image_path)
  
    # Definindo as opções do pipeline de OCR
    pipeline_options = PdfPipelineOptions()

    # Habilitar OCR
    pipeline_options.do_ocr = True

    # Habilitar estrutura de tabela (se necessário)
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.do_cell_matching = True

    # Definir o idioma do OCR (português)
    pipeline_options.ocr_options.lang = ["pt-br"]

    # Ativar a aceleração via CUDA
    pipeline_options.accelerator_options = AcceleratorDevice.CUDA

    # Criar o conversor de documentos com as opções configuradas
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
            )
        }
    )

    # Converter a imagem temporária para texto
    doc = converter.convert(temp_image_path).document
    md = doc.export_to_markdown()

    # Remover a imagem temporária
    temp_image_path.unlink()

    return md

In [4]:
def extrai_texto_OCR(item):   
    """
    Iterates over a list of source items, extracts metadata, opens PDF documents, and processes each page as a black and white image with 300 DPI.
    Args:
        source (list): List of file paths to PDF documents.
    Variables:
        lista_md (list): List to store metadata and extracted text.
        numero_tj (str): Extracted metadata representing the document number.
        tipo_doc (str): Extracted metadata representing the document type.
        pdf_document (fitz.Document): PDF document object.
        md_blocks (list): List to store markdown text blocks for each page.
        page (fitz.Page): Page object of the PDF document.
        pix (fitz.Pixmap): Pixmap object representing the page image.
        image (PIL.Image): Image object created from the pixmap.
        bw_image (PIL.Image): Black and white version of the image.
    Note:
        The code for extracting text and converting it to markdown is commented out.
        The code for appending the markdown text to the list and creating a DataFrame is also commented out.
    """
    # Cria a a variável para armazenar o tempo inicial
    ini = time.time()

    print(item)

    # extrai metadados
    numero_tj = item.split("\\")[1].split("_")[0]
    tipo_doc = item.split('\\')[1].split("_",1)[1].split(".")[0]

    # Abrir o documento PDF
    pdf_document = fitz.open(item.replace("\\", "/"))
    # Extrai o total de páginas
    total_paginas = int(pdf_document.page_count)

    # Salvar cada página como uma imagem em preto e branco com 300 DPI

    md_extract = ''
    for i in tqdm(range(len(pdf_document))):
        # faz a leitura da página do documento
        page = pdf_document.load_page(i)
        # Ajusta o valor do retângulo conforme necessário
        rect = fitz.Rect([20, 165, 550, 800])

        # pega a imagem da página em 300 DPI
        pix = page.get_pixmap(dpi=250, clip=rect)

        # converte a imagem em um objeto PIL
        image = pix.pil_image() # Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # # extrai texto e converte em markdown
        md_extract += extrai_texto_de_imagem(image)

        # salva o tempo final
        fim = time.time() - ini
        # cria a variavel do tempo de execução
        tempo_exec = fim - ini

    return numero_tj, tipo_doc, md_extract, total_paginas, tempo_exec # Retorna o texto extraído


In [7]:
dados = extrai_texto_OCR(source[0])


C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/sentença\0000049-51.2018.8.26.0603_sentença_pags_210-218.pdf


  0%|          | 0/9 [00:00<?, ?it/s]

C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\1 - criação_base\pasta_docs_tmp\temp_image.jpeg


 11%|█         | 1/9 [00:26<03:30, 26.33s/it]

C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\1 - criação_base\pasta_docs_tmp\temp_image.jpeg


...

 78%|███████▊  | 7/9 [03:30<01:00, 30.44s/it]

C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\1 - criação_base\pasta_docs_tmp\temp_image.jpeg


 89%|████████▉ | 8/9 [03:59<00:30, 30.20s/it]

C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\1 - criação_base\pasta_docs_tmp\temp_image.jpeg


100%|██████████| 9/9 [04:27<00:00, 29.74s/it]


In [8]:
for item in dados:
    print(item)


0000049-51.2018.8.26.0603
sentença_pags_210-218
Processo Digital n%: Classe Assunto: Documento de Origem:

0000049-51.2018.8.26.0603

Ação Penal Procedimento Ordinário Homicídio Simples CF , BO, IP 146/2018 DELSEC.ARAÇATUBA PLANTãO, 146/2018 - DELSEC.ARAÇATUBA PLANTãO, 003/2018 30 Distrito Policial de Araçatuba Justiça Pública

Autor:

Réu:

Juiz de Direito: Dr. Wellington José Prates

## VISTOS, etc.

foi denunciado como incurso nos arts..306 e 309, ambos da Lei n 9.303/97 (Código de Trânsito Brasileiro), em concurso formal (art.70), e no inc. V e VII, C.C. art.14, inc.II, em concurso material com aqueles (art.69) todos do Código Penal, porque, segundo a denúncia; n dia 04/jan/2018, por volta das 14h25, n0 cruzamento das ruas José Guerra e Rotary Club, bairro Jardim Moreira, nesta comarca, conduzia a motocicleta Honda CG 125 TODAY, placa BFF-9992/Araçatuba SP , em via pública; sem a devida habilitação para dirigir; gerando perigo de dano, e com sua capacidade psicomotora alterada em r

In [ ]:
# for item in source:
#     # Abrir o documento PDF
#     pdf_document = fitz.open(item.replace("\\","/"))

#     md_extract = ''
#     for i in tqdm(range(len(pdf_document))):
#         # faz a leitura da página do documento
#         page = pdf_document.load_page(i)
#         # Ajusta o valor do retângulo conforme necessário
#         rect = fitz.Rect([10, 137, 550, 750])

#         # pega a imagem da página em 300 DPI
#         pix = page.get_pixmap(dpi=300, clip=rect)

#         # converte a imagem em um objeto PIL
#         image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

#         # Salvar a imagem temporariamente para processamento
#         temp_image_path = Path("temp_image.jpeg")
#         image.save(temp_image_path)

#         # extrai texto e converte em markdown
#         md_extract += extrai_texto_de_imagem(temp_image_path)

#         # Remover a imagem temporária
#         temp_image_path.unlink()

#     print(md_extract)